# Neural Network from Scratch: XOR Classifier

## Modules 1–2 · Python & PyTorch

This notebook builds a **2-layer neural network in PyTorch** that learns the XOR function.

### Assignment Requirements
- 2 input neurons → 4 hidden neurons → 1 output neuron
- ReLU hidden activation
- Sigmoid output activation
- XOR dataset as PyTorch tensors
- `nn.BCELoss`
- Adam optimizer with learning rate `0.01`
- Manual training loop for 5000 epochs
- Print loss every 500 epochs
- Test all four XOR inputs
- Demonstrate edge-case handling
- Explain why a single-layer perceptron cannot solve XOR


## 1. Import Required Libraries

We use:
- **PyTorch** for tensors and neural-network operations
- **torch.nn** for layers and loss functions
- **torch.optim** for the Adam optimizer


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)


## 2. Reproducibility

Neural-network weights are initialized randomly. Setting a seed makes the experiment more reproducible.


In [ ]:
torch.manual_seed(42)
print("Random seed set to 42")


## 3. Create the XOR Dataset

The XOR truth table is:

| Input A | Input B | Target |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |


In [ ]:
X = torch.tensor([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
], dtype=torch.float32)

y = torch.tensor([
    [0.],
    [1.],
    [1.],
    [0.]
], dtype=torch.float32)

print("Inputs:")
print(X)

print("\nTargets:")
print(y)

print("\nInput shape:", X.shape)
print("Target shape:", y.shape)


## 4. Define the Neural Network

Architecture:

**2 inputs → 4 hidden neurons → ReLU → 1 output → Sigmoid**

The hidden layer introduces non-linearity, which is essential because XOR is not linearly separable.


In [ ]:
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.hidden = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.output = nn.Linear(4, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        x = self.sigmoid(x)
        return x


model = XORNet()
print(model)


## 5. Loss Function and Optimizer

- `nn.BCELoss()` measures binary classification error.
- Adam updates the network weights.
- Learning rate = `0.01`, as required by the assignment.


In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", 0.01)


## 6. Manual Training Loop

Each epoch performs:
1. Forward pass
2. Loss calculation
3. Gradient reset
4. Backpropagation
5. Weight update

The loss is printed every 500 epochs.


In [ ]:
epochs = 5000
loss_history = []

for epoch in range(epochs):
    # 1. Forward pass
    outputs = model(X)

    # 2. Calculate loss
    loss = criterion(outputs, y)

    # Edge case: detect invalid loss
    if torch.isnan(loss) or torch.isinf(loss):
        print("Training stopped: loss became NaN or infinite.")
        break

    # 3. Clear old gradients
    optimizer.zero_grad()

    # 4. Backpropagation
    loss.backward()

    # 5. Update weights
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 500 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {loss.item():.6f}")

print("\nTraining completed.")


## 7. Plot Training Loss

A decreasing loss indicates that the model is learning the XOR mapping.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(loss_history) + 1), loss_history)
plt.xlabel("Epoch")
plt.ylabel("Binary Cross Entropy Loss")
plt.title("XOR Model Training Loss")
plt.grid(True)
plt.show()


## 8. Safe Prediction Function with Edge-Case Handling

This function demonstrates practical robustness:

- Checks that input has shape `[N, 2]`
- Checks for NaN values
- Checks for infinite values
- Checks for empty model output
- Uses `try/except` to report unexpected errors


In [ ]:
def predict_xor(test_input):
    try:
        # Convert input to a PyTorch tensor
        test_input = torch.tensor(test_input, dtype=torch.float32)

        # Edge case 1: incorrect shape
        if test_input.ndim != 2 or test_input.shape[1] != 2:
            raise ValueError(
                "Input must have shape [N, 2]. Example: [[0, 1]]"
            )

        # Edge case 2: NaN values
        if torch.isnan(test_input).any():
            raise ValueError("Input contains NaN values.")

        # Edge case 3: infinite values
        if torch.isinf(test_input).any():
            raise ValueError("Input contains infinite values.")

        # Inference does not need gradients
        with torch.no_grad():
            prediction = model(test_input)

            # Edge case 4: empty output
            if prediction.numel() == 0:
                raise ValueError("Model returned an empty output.")

            predicted_class = (prediction >= 0.5).float()

        print("Prediction Results")
        print("-" * 60)

        for i in range(len(test_input)):
            print(
                f"Input: {test_input[i].tolist()} | "
                f"Probability: {prediction[i].item():.4f} | "
                f"Class: {int(predicted_class[i].item())}"
            )

        return prediction, predicted_class

    except Exception as e:
        print("Prediction Error:", e)
        return None, None


## 9. Test All Four XOR Inputs

The trained model should predict:

- `[0, 0]` → `0`
- `[0, 1]` → `1`
- `[1, 0]` → `1`
- `[1, 1]` → `0`


In [ ]:
predictions, predicted_classes = predict_xor([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])


## 10. Verify Correctness Automatically

This cell compares the predicted classes with the expected XOR targets and reports accuracy.


In [ ]:
with torch.no_grad():
    final_probabilities = model(X)
    final_classes = (final_probabilities >= 0.5).float()

accuracy = (final_classes == y).float().mean().item() * 100

print("Expected:", y.squeeze().tolist())
print("Predicted:", final_classes.squeeze().tolist())
print(f"Accuracy: {accuracy:.2f}%")

if accuracy == 100.0:
    print("SUCCESS: The model correctly predicts all 4 XOR outputs.")
else:
    print("The model did not reach 100% accuracy. Consider retraining or adjusting hyperparameters.")


## 11. Edge Case Test: Wrong Input Shape

The network requires two features per sample. This deliberately invalid input should produce a clear error instead of an unexplained model failure.


In [ ]:
predict_xor([[0, 1, 0]])


## 12. Edge Case Test: NaN Input

A NaN value means **Not a Number**. The prediction function detects it before sending the input through the model.


In [ ]:
predict_xor([[float("nan"), 1]])


## 13. Edge Case Test: Infinite Input

Infinite values are also rejected before inference.


In [ ]:
predict_xor([[float("inf"), 1]])


## 14. Why a Single-Layer Perceptron Cannot Solve XOR

A single-layer perceptron can learn only a **linear decision boundary**.

The XOR classes are:

- Class 0: `(0,0)` and `(1,1)`
- Class 1: `(0,1)` and `(1,0)`

There is no single straight line that separates these two classes correctly.

The hidden layer plus ReLU creates a non-linear transformation of the input. This allows the two-layer network to learn a non-linear decision boundary and solve XOR.

This is the key lesson: **neural-network depth and non-linear activations allow models to learn patterns that a single linear layer cannot represent.**


## 15. Real-World Connection

Although XOR is a tiny educational dataset, the same learning process is used in real applications.

For example, in banking, a prediction may depend on multiple interacting factors such as income, credit history, repayment behavior, and existing obligations. These relationships may not be linearly separable.

This XOR project therefore provides a foundation for understanding more complex applications such as fraud detection, credit-risk prediction, customer classification, image recognition, and other machine-learning systems.


## 16. Edge Case Awareness — Assignment Explanation

### Potential failure points

1. **Malformed input:** The model expects `[N, 2]` numerical input. Shape validation prevents invalid inputs from reaching the model.
2. **NaN or infinite values:** These are explicitly detected and rejected.
3. **Invalid training loss:** If the loss becomes NaN or infinite, training is stopped.
4. **Empty output:** The prediction function checks that the model returned values.
5. **Random initialization:** `torch.manual_seed(42)` makes the experiment reproducible.
6. **Missing PyTorch installation:** The project should include `requirements.txt` and README setup instructions.

These checks make the implementation more robust and demonstrate awareness of real-world failure conditions.


## 17. Final Assignment Checklist

- [x] PyTorch neural network
- [x] 2 input neurons
- [x] 4 hidden neurons
- [x] 1 output neuron
- [x] ReLU activation
- [x] Sigmoid output
- [x] XOR tensors
- [x] BCELoss
- [x] Adam optimizer
- [x] Learning rate 0.01
- [x] 5000 training epochs
- [x] Loss printed every 500 epochs
- [x] Final predictions for all 4 inputs
- [x] Accuracy verification
- [x] Training-loss visualization
- [x] Edge-case handling
- [x] Explanation of non-linear decision boundaries
- [x] Real-world/business connection
